# Canifa - Sinh caption cho anh san pham (3 anh -> 1 caption)
## Ban Qwen2-VL-7B-Instruct (chay local tren GPU, khong can API key / khong bi rate limit)

- Model **Qwen2-VL-7B-Instruct** duoc tai ve va chay TRUC TIEP tren GPU cua Kaggle (T4 x2), quantize 4-bit (NF4 qua bitsandbytes) de vua VRAM ~15GB cua 1 GPU T4.


## 1. Cai dat thu vien

In [ ]:
!pip install -q -U "transformers>=4.49.0" accelerate bitsandbytes qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 89.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 56.9 MB/s eta 0:00:00:00:0100:01


## 2. Import, logging, seed, config

In [2]:
# --- stdlib ---
import gc
import io
import json
import logging
import os
import random
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

# --- third-party ---
import torch
from PIL import Image
from tqdm.auto import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("caption_synthesis")


def set_seed(seed: int) -> None:
    """Co dinh seed cho random + torch (giup output on dinh hon khi do_sample=True)."""
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_hf_token(secret_name: str = "HF_TOKEN") -> Optional[str]:
    """Lay HF token tu Kaggle Secrets (tuy chon - Qwen2-VL-7B-Instruct la model public
    nen KHONG bat buoc phai co token, chi giup tai nhanh/on dinh hon).

    Raises:
        Khong raise - neu khong tim thay secret, tra ve None va notebook van chay binh thuong.
    """
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(secret_name)
    except ImportError:
        return os.getenv(secret_name)
    except Exception as e:
        logger.info("Khong tim thay secret '%s' (khong bat buoc): %s", secret_name, e)
        return os.getenv(secret_name)


In [3]:
@dataclass(frozen=True)
class PathConfig:
    image_dir: Path
    metadata_csv: Path          # CSV crawl goc dang WIDE: product_slug, caption (goc), image_1_local_path...
    output_jsonl: Path          # noi ghi ket qua, append-only de resume duoc
    final_csv: Path


@dataclass(frozen=True)
class ModelConfig:
    model_name: str              # "Qwen/Qwen2-VL-7B-Instruct"
    quant_mode: str = "bnb4bit"  # "bnb4bit" | "none" (none = fp16 full, can >15GB VRAM, khong khuyen nghi tren 1 T4)
    device: str = "cuda:0"       # GPU dau tien trong T4 x2 (xem phan "Tuy chon nang cao" de dung ca 2 GPU)
    max_image_side: int = 1024   # resize anh truoc khi dua vao model (giam VRAM + toc do)
    min_pixels: int = 256 * 28 * 28   # gioi han so "visual token" toi thieu / anh (Qwen2-VL dynamic resolution)
    max_pixels: int = 1024 * 28 * 28  # gioi han toi da - cang lon cang ton VRAM & cham hon
    max_new_tokens: int = 220    # du cho 2-3 cau tieng Viet
    do_sample: bool = True       # True -> caption da dang hon giua cac san pham; False -> greedy, on dinh 100%
    temperature: float = 0.7
    top_p: float = 0.9
    repetition_penalty: float = 1.05


@dataclass(frozen=True)
class RunConfig:
    seed: int = 42
    n_images_per_product: int = 3
    max_retries: int = 3
    backoff_base_seconds: float = 2.0
    checkpoint_every: int = 20

    paths: PathConfig = field(default_factory=lambda: PathConfig(
        # TODO: doi lai duong dan dataset ban attach vao notebook Kaggle nay
        image_dir=Path("/kaggle/input/datasets/qa9999/canifa/canifa_images/images"),
        metadata_csv=Path("/kaggle/input/datasets/qa9999/canifa/canifa_products_wide.csv"),
        output_jsonl=Path("/kaggle/working/captions_synth.jsonl"),
        final_csv=Path("/kaggle/working/canifa_captions_final.csv"),
    ))

    model: ModelConfig = field(default_factory=lambda: ModelConfig(
        model_name="Qwen/Qwen2-VL-7B-Instruct",
    ))


cfg = RunConfig()
set_seed(cfg.seed)
cfg.paths.output_jsonl.parent.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError(
        "Khong tim thay GPU. Vao Settings > Accelerator, chon GPU T4 x2 roi Restart & Run All."
    )
logger.info("GPU dang dung: %s", torch.cuda.get_device_name(0))


2026-07-16 22:06:48,107 | INFO | caption_synthesis | GPU dang dung: Tesla T4


## 3. Quet & group anh theo san pham

Pattern file: `{product_id}_{1|2|3}.webp`. Chay debug 1 san pham TRUOC khi group toan bo,
theo dung thoi quen validate tung phan cua ban.


In [4]:
import pandas as pd
from pathlib import PureWindowsPath

IMAGE_COLUMNS = ["image_1_local_path", "image_2_local_path", "image_3_local_path"]
REQUIRED_COLUMNS = ["product_slug", "caption", *IMAGE_COLUMNS]


def load_metadata(metadata_csv: Path) -> pd.DataFrame:
    """Doc CSV metadata dang wide (1 dong = 1 san pham, kem san 3 cot duong dan anh).

    Raises:
        FileNotFoundError: neu khong tim thay file CSV.
        KeyError: neu CSV thieu cot bat buoc.
    """
    if not metadata_csv.exists():
        raise FileNotFoundError(f"Khong tim thay metadata CSV: {metadata_csv}")

    df = pd.read_csv(metadata_csv)
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        raise KeyError(f"CSV thieu cac cot bat buoc: {missing_cols}")

    return df


def resolve_image_path(local_path_windows: str, image_dir: Path) -> Path:
    """Lay ten file tu duong dan Windows goc (vi du D:\\canifa_dataset\\images\\xxx.webp)
    va noi voi image_dir thuc te tren Kaggle.

    Ly do can ham nay: cot `image_N_local_path` trong CSV la duong dan tuyet doi tren may
    Windows cua nguoi crawl, khong ton tai tren Kaggle - chi phan ten file la con dung duoc.
    """
    filename = PureWindowsPath(str(local_path_windows)).name
    return image_dir / filename


def build_product_records(df: pd.DataFrame, image_dir: Path,
                           expected_images: int = 3) -> dict[str, dict]:
    """Group theo product_slug, resolve duong dan anh thuc te tren Kaggle, validate ton tai tren dia.

    Returns:
        Mapping product_slug -> {"image_paths": [Path, ...], "original_caption": str}.
        `original_caption` la caption GOC tu CSV (co the chua thong tin khong nhin thay duoc tu anh
        nhu chat lieu vai) - CHI de doi chieu/tham khao, KHONG duoc feed vao prompt cho model.
    """
    if not image_dir.exists():
        raise FileNotFoundError(f"Khong tim thay thu muc anh: {image_dir}")

    records: dict[str, dict] = {}
    missing_files: list[str] = []

    for _, row in df.iterrows():
        pid = row["product_slug"]
        image_paths = []
        for col in IMAGE_COLUMNS[:expected_images]:
            p = resolve_image_path(row[col], image_dir)
            if not p.exists():
                missing_files.append(str(p))
                continue
            image_paths.append(p)

        if len(image_paths) != expected_images:
            logger.warning("San pham %s thieu anh (%d/%d), bo qua.", pid, len(image_paths), expected_images)
            continue

        records[pid] = {"image_paths": image_paths, "original_caption": row["caption"]}

    if missing_files:
        logger.warning("Co %d file anh khong tim thay tren dia (xem chi tiet trong log).", len(missing_files))

    logger.info("Da group duoc %d san pham hop le / %d dong CSV.", len(records), len(df))
    return records


metadata_df = load_metadata(cfg.paths.metadata_csv)
product_records = build_product_records(metadata_df, cfg.paths.image_dir,
                                         expected_images=cfg.n_images_per_product)


2026-07-16 22:06:48,240 | INFO | numexpr.utils | NumExpr defaulting to 4 threads.
2026-07-16 22:06:50,790 | WARNING | caption_synthesis | San pham khau-trang-unisex-nguoi-lon-5ao26s003-sk010 thieu anh (2/3), bo qua.
2026-07-16 22:06:50,796 | WARNING | caption_synthesis | San pham khan-da-nang-5ac26s001-sk010 thieu anh (2/3), bo qua.
2026-07-16 22:06:50,803 | WARNING | caption_synthesis | San pham khan-da-nang-5ac26s001-sb228 thieu anh (2/3), bo qua.
2026-07-16 22:06:50,809 | WARNING | caption_synthesis | San pham khan-da-nang-5ac26s001-sb001 thieu anh (2/3), bo qua.
2026-07-16 22:06:51,021 | WARNING | caption_synthesis | San pham bang-co-tay-5ao26s004-sw001 thieu anh (1/3), bo qua.
2026-07-16 22:07:00,516 | WARNING | caption_synthesis | San pham bo-quan-ao-be-gai-1st25w007-sw011 thieu anh (1/3), bo qua.
2026-07-16 22:07:02,808 | WARNING | caption_synthesis | San pham quan-ni-be-trai-2bp25w004 thieu anh (2/3), bo qua.
2026-07-16 22:07:04,690 | WARNING | caption_synthesis | Co 9 file anh

## 4. Load model Qwen2-VL-7B-Instruct (quantize 4-bit) len GPU

Model ~7B tham so, quantize NF4 4-bit qua `bitsandbytes` chi con khoang **5-6GB VRAM** cho trong so
(chua ke KV-cache/activation khi generate) -> vua thoai mai trong 15GB cua 1 GPU T4. Neu ban muon
dung ban AWQ chinh chu cua Qwen (`Qwen/Qwen2-VL-7B-Instruct-AWQ`, can cai them `autoawq`) thi doi
`quant_mode="awq"` va sua lai phan load model o duoi (co comment huong dan san).

Chi chay cell nay **1 lan** - load lai se ton thoi gian va VRAM.


In [5]:
!pip install -U bitsandbytes

In [6]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_NAME = cfg.model.model_name

if cfg.model.quant_mode == "bnb4bit":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,   # T4 (Turing) khong co bf16 tensor core -> dung fp16
        bnb_4bit_use_double_quant=True,
    )
elif cfg.model.quant_mode == "none":
    quant_config = None
else:
    raise ValueError(f"quant_mode khong ho tro: {cfg.model.quant_mode}")

# --- Neu muon dung ban AWQ chinh chu cua Qwen thay vi bnb 4-bit, uncomment 2 dong duoi va
#     doi MODEL_NAME + cai them `!pip install -q autoawq`. Khi dung AWQ thi KHONG can
#     truyen quantization_config vi trong so da duoc quantize san trong checkpoint.
# MODEL_NAME = "Qwen/Qwen2-VL-7B-Instruct-AWQ"
# quant_config = None

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=cfg.model.min_pixels,
    max_pixels=cfg.model.max_pixels,
    token=get_hf_token(),
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map=cfg.model.device,   # 1 GPU duy nhat ("cuda:0") - xem phan Tuy chon nang cao de dung ca 2 GPU
    attn_implementation="sdpa",    # an toan tren moi may Kaggle; doi "flash_attention_2" neu da cai flash-attn
    token=get_hf_token(),
)
model.eval()

logger.info("Da load model %s (quant_mode=%s) len %s", MODEL_NAME, cfg.model.quant_mode, cfg.model.device)
logger.info("VRAM da cap phat: %.2f GB", torch.cuda.memory_allocated() / 1e9)


2026-07-16 22:07:24,376 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-16 22:07:24,463 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 22:07:24,483 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-VL-7B-Instruct/eed13092ef92e448dd6875b2a00151bd3f7db0ac/preprocessor_config.json "HTTP/1.1 200 OK"
2026-07-16 22:07:24,592 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-VL-7B-Instruct/eed13092ef92e448dd6875b2a00151bd3f7db0ac/preprocessor_config.json "HTTP/1.1 200 OK"


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

2026-07-16 22:07:24,705 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-16 22:07:24,796 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 22:07:24,809 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-VL-7B-Instruct/eed13092ef92e448dd6875b2a00151bd3f7db0ac/preprocessor_config.json "HTTP/1.1 200 OK"
2026-07-16 22:07:24,902 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2-VL-7B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-07-16 22:07:24,988 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-16 22:07:25,078 | INFO | httpx | HTTP Request:

chat_template.json: 0.00B [00:00, ?B/s]

2026-07-16 22:07:25,225 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-07-16 22:07:25,319 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/main/audio_tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-07-16 22:07:25,408 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-16 22:07:25,495 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 22:07:25,509 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-VL-7B-Instruct/eed13092ef92e448dd6875b2a00151bd3f7db0ac/preprocessor_config.json "HTTP/1.1 200 OK"
2026-07-16 22:07:25,600 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qw

...

2026-07-16 22:07:33,405 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/eed13092ef92e448dd6875b2a00151bd3f7db0ac/model-00001-of-00005.safetensors "HTTP/1.1 302 Found"
2026-07-16 22:07:33,447 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/eed13092ef92e448dd6875b2a00151bd3f7db0ac/model-00004-of-00005.safetensors "HTTP/1.1 302 Found"
2026-07-16 22:07:33,453 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/eed13092ef92e448dd6875b2a00151bd3f7db0ac/model-00003-of-00005.safetensors "HTTP/1.1 302 Found"
2026-07-16 22:07:33,455 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/eed13092ef92e448dd6875b2a00151bd3f7db0ac/model-00005-of-00005.safetensors "HTTP/1.1 302 Found"
2026-07-16 22:07:33,458 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/eed13092ef92e448dd6875b2a00151bd3f7

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

2026-07-16 22:10:10,507 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 22:10:10,618 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-VL-7B-Instruct/eed13092ef92e448dd6875b2a00151bd3f7db0ac/generation_config.json "HTTP/1.1 200 OK"
2026-07-16 22:10:10,725 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-VL-7B-Instruct/eed13092ef92e448dd6875b2a00151bd3f7db0ac/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

2026-07-16 22:10:10,742 | INFO | caption_synthesis | Da load model Qwen/Qwen2-VL-7B-Instruct (quant_mode=bnb4bit) len cuda:0
2026-07-16 22:10:10,743 | INFO | caption_synthesis | VRAM da cap phat: 5.93 GB


## 5. Prompt & ham goi model (3 anh cung luc -> 1 caption)

In [7]:
@dataclass
class CaptionResult:
    product_id: str
    caption: Optional[str]
    provider: str
    model_name: str
    success: bool
    error: Optional[str] = None


CAPTION_PROMPT = """
You are an expert fashion copywriter specializing in creating product descriptions for major e-commerce platforms like Shopee, Lazada, and TikTok Shop.
Your task is to write a short, engaging, and highly persuasive product description (caption) in Vietnamese (exactly 2 to 3 sentences) based on the 3 provided images of the SAME fashion item.

<CONTENT_REQUIREMENTS>
1. Visual Description (80%): Accurately describe the garment type, fit/silhouette (e.g., loose-fit, slim-fit, cropped), colors (primary and secondary/accent colors), and visible design details (e.g., unique patterns, necklines, pockets, buttons, zippers, ribbed cuffs).
2. Styling & Occasion (20%): Provide 1 practical styling tip (how to mix-and-match) or suggest a suitable occasion (e.g., school, work, casual hangouts, street style) based on the product's overall vibe and aesthetic.
</CONTENT_REQUIREMENTS>

<TONE_AND_STYLE>
- Write in a natural, youthful, friendly, and highly engaging Vietnamese tone that appeals to online shoppers.
- Avoid dry, technical, or inventory-like reporting. Instead of writing "áo màu xanh cổ tròn", make it sound more attractive like "sở hữu gam màu xanh tươi tắn cùng thiết kế cổ tròn basic...".
</TONE_AND_STYLE>

<STRICT_NEGATIVE_CONSTRAINTS>
- DO NOT guess or assume the fabric/material (e.g., do NOT write "cotton 100%", "linen", "polyester" unless a fabric tag is clearly visible). You may only describe visual texture or drape if highly obvious (e.g., "chất vải mềm mại", "đứng form").
- DO NOT use over-the-top, cliché marketing buzzwords that sound spammy (e.g., do NOT use "tự tin tỏa sáng", "đẳng cấp", "thời thượng", "cao cấp nhất"). Keep it premium yet realistic.
- DO NOT use any Markdown formatting, including bolding (**), italics (*), or bullet points (-). The output must be clean, plain text.
- DO NOT include any introductory or concluding remarks (e.g., do NOT write "Đây là mô tả sản phẩm:" or "Hy vọng bạn thích..."). Return ONLY the 2-3 sentence Vietnamese paragraph.
</STRICT_NEGATIVE_CONSTRAINTS>
"""


def load_and_resize(path: Path, max_side: int) -> Image.Image:
    """Doc anh tu dia va resize (neu can) - giam VRAM/token khi dua vao model.

    Raises:
        FileNotFoundError: neu anh khong ton tai.
    """
    if not path.exists():
        raise FileNotFoundError(f"Khong tim thay anh: {path}")
    img = Image.open(path).convert("RGB")
    if max(img.size) > max_side:
        ratio = max_side / max(img.size)
        img = img.resize((int(img.width * ratio), int(img.height * ratio)), Image.LANCZOS)
    return img


def build_messages(image_paths: list[Path], max_image_side: int) -> list[dict]:
    """Dung dinh dang 'chat' cua Qwen2-VL: nhieu block anh + 1 block text trong CUNG 1 message.
    Day chinh la co che giup model doc DONG THOI 3 anh va sinh 1 caption chung cho ca 3.
    """
    content = [{"type": "image", "image": load_and_resize(p, max_image_side)} for p in image_paths]
    content.append({"type": "text", "text": CAPTION_PROMPT})
    return [{"role": "user", "content": content}]


@torch.inference_mode()
def call_qwen(image_paths: list[Path], model_cfg: ModelConfig) -> str:
    """Goi model local (khong qua internet, khong rate limit) de sinh caption tu 3 anh.

    Raises:
        RuntimeError: neu model tra ve chuoi rong (hiem gap, thuong do generate bi cat som).
    """
    from qwen_vl_utils import process_vision_info

    messages = build_messages(image_paths, model_cfg.max_image_side)
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=model_cfg.max_new_tokens,
        do_sample=model_cfg.do_sample,
        temperature=model_cfg.temperature if model_cfg.do_sample else None,
        top_p=model_cfg.top_p if model_cfg.do_sample else None,
        repetition_penalty=model_cfg.repetition_penalty,
    )
    trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(trimmed, skip_special_tokens=True,
                                          clean_up_tokenization_spaces=False)[0].strip()

    if not output_text:
        raise RuntimeError("Model tra ve chuoi rong.")
    return output_text


## 6. TEST 1 SAN PHAM (bat buoc chay truoc khi chay full)

Chay cell duoi de xac nhan: model load dung, output la tieng Viet, dung 2-3 cau, khong dinh
markdown/buzzword cam. Neu output chua on, chinh `do_sample`, `temperature`, `max_new_tokens`
trong `ModelConfig` roi chay lai cell nay (KHONG can load lai model).

In [10]:
sample_pid = next(iter(product_records))
sample = product_records[sample_pid]
print("product_slug:", sample_pid)
test_caption = call_qwen(sample["image_paths"], cfg.model)
print("=== CAPTION SINH RA (Qwen2-VL) ===")
print(test_caption)
print()
print("=== Caption goc tu CSV (chi de doi chieu) ===")
print(sample["original_caption"])


product_slug: ao-phong-unisex-ngưoi-lon-5ts26s006-sl267
=== CAPTION SINH RA (Qwen2-VL) ===
Áo thun màu xanh đậm này có thiết kế cổ tròn và cổ áo rộng rãi, phù hợp với nhiều phong cách thời trang. Bạn có thể kết hợp nó với quần jeans hoặc short, tạo nên một look casual hoàn hảo cho ngày hè.

=== Caption goc tu CSV (chi de doi chieu) ===
Áo phông unisex người lớn, thiết kế dáng rộng, hình in Mickey & Friends trẻ trung, phù hợp đa dạng phong cách.Sản phẩm kết hợp 60% cotton và 40% polyester, giúp vải xốp nhẹ, giữ phom và tăng độ bền.


## 7. Vong lap chinh - co resume + checkpoint

Khac voi ban Gemini: KHONG can RateLimiter / xoay vong API key nua vi model chay local.
Retry o day chi xu ly loi ky thuat (het VRAM tam thoi, output rong...), khong lien quan quota.

In [11]:
def load_already_processed(output_path: Path) -> set[str]:
    """Doc cac product_slug da xu ly de resume, tranh chay lai lang phi thoi gian GPU."""
    if not output_path.exists():
        return set()
    processed = set()
    with output_path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)
                processed.add(record["product_slug"])
            except (json.JSONDecodeError, KeyError) as e:
                logger.error("Dong loi trong file output, bo qua: %s", e)
    return processed


def generate_with_retry(pid: str, image_paths: list[Path], model_cfg: ModelConfig,
                         max_retries: int, backoff_base: float) -> CaptionResult:
    """Goi call_qwen voi retry cho cac loi ky thuat (het VRAM tam thoi, output rong, v.v.).
    Khong co khai niem 'het quota' o day vi model chay local tren GPU cua chinh minh.
    """
    last_error: Optional[str] = None

    for attempt in range(1, max_retries + 1):
        try:
            caption = call_qwen(image_paths, model_cfg)
            return CaptionResult(pid, caption, "qwen2vl-local", model_cfg.model_name, True)
        except torch.cuda.OutOfMemoryError as e:
            last_error = f"CUDA OOM (lan {attempt}): {e}"
            logger.warning("San pham %s: %s. Don VRAM va thu lai...", pid, last_error)
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(backoff_base * attempt)
        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"
            logger.warning("San pham %s loi (lan %d/%d): %s", pid, attempt, max_retries, last_error)
            time.sleep(backoff_base * attempt)

    return CaptionResult(pid, None, "qwen2vl-local", model_cfg.model_name, False, last_error)


def run_synthesis(records: dict[str, dict], cfg: RunConfig) -> None:
    """Sinh caption cho tung san pham trong `records`, ghi append-only vao output_jsonl.

    `records` co dang {product_slug: {"image_paths": [...], "original_caption": str}},
    dung dinh dang tra ve boi `build_product_records`.
    """
    processed = load_already_processed(cfg.paths.output_jsonl)
    remaining = {pid: r for pid, r in records.items() if pid not in processed}

    logger.info("Da xu ly truoc do: %d | Con lai: %d", len(processed), len(remaining))

    with cfg.paths.output_jsonl.open("a", encoding="utf-8") as f:
        for i, (pid, record) in enumerate(tqdm(remaining.items()), start=1):
            result = generate_with_retry(pid, record["image_paths"], cfg.model,
                                          cfg.max_retries, cfg.backoff_base_seconds)
            f.write(json.dumps({
                "product_slug": result.product_id,
                "caption_synth": result.caption,
                "provider": result.provider,
                "model_name": result.model_name,
                "success": result.success,
                "error": result.error,
            }, ensure_ascii=False) + "\n")

            if i % cfg.checkpoint_every == 0:
                f.flush()
                logger.info("Checkpoint: da xu ly %d/%d", i, len(remaining))


In [12]:
# Chi bo comment dong duoi SAU KHI da xac nhan test_caption o Cell 13 hop ly:
run_synthesis(product_records, cfg)


2026-07-16 22:22:12,335 | INFO | caption_synthesis | Da xu ly truoc do: 0 | Con lai: 1876


  0%|          | 0/1876 [00:00<?, ?it/s]

2026-07-16 22:27:25,753 | INFO | caption_synthesis | Checkpoint: da xu ly 20/1876
2026-07-16 22:32:44,695 | INFO | caption_synthesis | Checkpoint: da xu ly 40/1876
2026-07-16 22:38:26,571 | INFO | caption_synthesis | Checkpoint: da xu ly 60/1876
2026-07-16 22:43:52,005 | INFO | caption_synthesis | Checkpoint: da xu ly 80/1876
2026-07-16 22:49:14,687 | INFO | caption_synthesis | Checkpoint: da xu ly 100/1876
2026-07-16 22:55:20,318 | INFO | caption_synthesis | Checkpoint: da xu ly 120/1876
2026-07-16 23:01:35,185 | INFO | caption_synthesis | Checkpoint: da xu ly 140/1876
2026-07-16 23:07:12,853 | INFO | caption_synthesis | Checkpoint: da xu ly 160/1876
2026-07-16 23:12:35,997 | INFO | caption_synthesis | Checkpoint: da xu ly 180/1876
2026-07-16 23:18:02,599 | INFO | caption_synthesis | Checkpoint: da xu ly 200/1876
2026-07-16 23:23:57,962 | INFO | caption_synthesis | Checkpoint: da xu ly 220/1876
2026-07-16 23:28:55,634 | INFO | caption_synthesis | Checkpoint: da xu ly 240/1876
2026-07-

## 8. Merge ket qua voi metadata goc

In [13]:
def merge_with_metadata(output_jsonl: Path, metadata_csv: Path) -> pd.DataFrame:
    """Merge caption_synth sinh duoc voi metadata crawl goc theo product_slug.

    Ket qua se co CA HAI cot: `caption` (goc, tu crawl - co the chua chat lieu/BST khong
    nhin thay tu anh) va `caption_synth` (moi, sinh tu anh) - de tien doi chieu o buoc filtering.

    Raises:
        FileNotFoundError: neu output_jsonl hoac metadata_csv chua ton tai.
    """
    if not output_jsonl.exists():
        raise FileNotFoundError(f"Chua co file output caption: {output_jsonl}. Chay run_synthesis truoc.")
    if not metadata_csv.exists():
        raise FileNotFoundError(f"Khong tim thay metadata: {metadata_csv}")

    captions_df = pd.read_json(output_jsonl, lines=True)
    meta_df = pd.read_csv(metadata_csv)
    merged = meta_df.merge(captions_df, on="product_slug", how="left")

    n_missing = merged["caption_synth"].isna().sum()
    if n_missing:
        logger.warning("%d san pham chua co caption_synth (fail sau khi het retry hoac chua chay)",
                        n_missing)

    return merged


# Chi chay sau khi da run_synthesis xong (hoac chay 1 phan de kiem tra):
final_df = merge_with_metadata(cfg.paths.output_jsonl, cfg.paths.metadata_csv)
final_df.to_csv(cfg.paths.final_csv, index=False)
final_df.head()


2026-07-17 07:22:27,673 | WARNING | caption_synthesis | 7 san pham chua co caption_synth (fail sau khi het retry hoac chua chay)


,source,gender_category,category1,category1_name_vi,product_slug,product_name,sku,price,special_price,product_url,...,image_1_local_path,image_2_url,image_2_local_path,image_3_url,image_3_local_path,caption_synth,provider,model_name,success,error
0,canifa,nu,ao-phong,Áo phông / Áo thun,ao-phong-unisex-ngưoi-lon-5ts26s006-sl267,Áo phông unisex người lớn dáng rộng hình in Mi...,5TS26S006-SL267,349000.0,199000.0,https://canifa.com/ao-phong-unisex-ngưoi-lon-5...,...,D:\canifa_dataset\images\ao-phong-unisex-ngưoi...,https://2885371169.e.cdneverest.net/catalog/pr...,D:\canifa_dataset\images\ao-phong-unisex-ngưoi...,https://2885371169.e.cdneverest.net/catalog/pr...,D:\canifa_dataset\images\ao-phong-unisex-ngưoi...,Áo thun màu xanh đậm này có thiết kế cổ tròn v...,qwen2vl-local,Qwen/Qwen2-VL-7B-Instruct,True,NaN
1,canifa,nu,ao-phong,Áo phông / Áo thun,ao-ba-lo-nu-6ta26s003-sm192,Áo sát nách active nữ,6TA26S003-SM192,249000.0,NaN,https://canifa.com/ao-ba-lo-nu-6ta26s003-sm192,...,D:\canifa_dataset\images\ao-ba-lo-nu-6ta26s003...,https://2885371169.e.cdneverest.net/catalog/pr...,D:\canifa_dataset\images\ao-ba-lo-nu-6ta26s003...,https://2885371169.e.cdneverest.net/catalog/pr...,D:\canifa_dataset\images\ao-ba-lo-nu-6ta26s003...,Áo vest màu hồng pastel này sẽ là sự lựa chọn ...,qwen2vl-local,Qwen/Qwen2-VL-7B-Instruct,True,NaN
2,canifa,nu,ao-phong,Áo phông / Áo thun,ao-ba-lo-nu-6ta26s003-sk010,Áo sát nách active nữ,6TA26S003-SK010,249000.0,NaN,https://canifa.com/ao-ba-lo-nu-6ta26s003-sk010,...,D:\canifa_dataset\images\ao-ba-lo-nu-6ta26s003...,https://2885371169.e.cdneverest.net/catalog/pr...,D:\canifa_dataset\images\ao-ba-lo-nu-6ta26s003...,https://2885371169.e.cdneverest.net/catalog/pr...,D:\canifa_dataset\images\ao-ba-lo-nu-6ta26s003...,1. Áo vest nữ màu đen thiết kế đơn giản nhưng ...,qwen2vl-local,Qwen/Qwen2-VL-7B-Instruct,True,NaN
3,canifa,nu,ao-phong,Áo phông / Áo thun,ao-ba-lo-nu-6ta26s003-sw001,Áo sát nách active nữ,6TA26S003-SW001,249000.0,NaN,https://canifa.com/ao-ba-lo-nu-6ta26s003-sw001,...,D:\canifa_dataset\images\ao-ba-lo-nu-6ta26s003...,https://2885371169.e.cdneverest.net/catalog/pr...,D:\canifa_dataset\images\ao-ba-lo-nu-6ta26s003...,https://2885371169.e.cdneverest.net/catalog/pr...,D:\canifa_dataset\images\ao-ba-lo-nu-6ta26s003...,Áo vest trắng này có thiết kế đơn giản nhưng đ...,qwen2vl-local,Qwen/Qwen2-VL-7B-Instruct,True,NaN
4,canifa,nu,ao-phong,Áo phông / Áo thun,ao-phong-nu-6ts26s019-sw001,Áo phông active nữ dáng ôm | Wicking,6TS26S019-SW001,299000.0,NaN,https://canifa.com/ao-phong-nu-6ts26s019-sw001,...,D:\canifa_dataset\images\ao-phong-nu-6ts26s019...,https://2885371169.e.cdneverest.net/catalog/pr...,D:\canifa_dataset\images\ao-phong-nu-6ts26s019...,https://2885371169.e.cdneverest.net/catalog/pr...,D:\canifa_dataset\images\ao-phong-nu-6ts26s019...,Áo trắng này có thiết kế đơn giản nhưng đầy cá...,qwen2vl-local,Qwen/Qwen2-VL-7B-Instruct,True,NaN


## 9. Upload ket qua len Hugging Face

In [14]:
from huggingface_hub import login, HfApi

hf_token = get_hf_token()
if not hf_token:
    raise RuntimeError("Khong tim thay secret HF_TOKEN - vao Add-ons > Secrets de tao truoc khi upload.")

login(token=hf_token)
api = HfApi()

# Dinh nghia user va ten repo dataset (ban co the thay doi 'canifa-captions' theo y muon)
repo_id = "qa994/canifa-captions"

api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

api.upload_file(
    path_or_fileobj=str(cfg.paths.output_jsonl),
    path_in_repo="captions_synth.jsonl",
    repo_id=repo_id,
    repo_type="dataset",
)

api.upload_file(
    path_or_fileobj=str(cfg.paths.final_csv),
    path_in_repo="canifa_captions_final.csv",
    repo_id=repo_id,
    repo_type="dataset",
)

print(f"Da tai ket qua len Hugging Face thanh cong tai: https://huggingface.co/datasets/{repo_id}")


2026-07-17 07:22:28,079 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
2026-07-17 07:22:28,476 | INFO | httpx | HTTP Request: POST https://huggingface.co/api/repos/create "HTTP/1.1 200 OK"
2026-07-17 07:22:28,610 | INFO | httpx | HTTP Request: POST https://huggingface.co/api/datasets/qa994/canifa-captions/preupload/main "HTTP/1.1 200 OK"
2026-07-17 07:22:29,278 | INFO | httpx | HTTP Request: POST https://huggingface.co/api/datasets/qa994/canifa-captions/commit/main "HTTP/1.1 200 OK"
2026-07-17 07:22:29,556 | INFO | httpx | HTTP Request: POST https://huggingface.co/api/datasets/qa994/canifa-captions/preupload/main "HTTP/1.1 200 OK"
2026-07-17 07:22:30,944 | INFO | httpx | HTTP Request: POST https://huggingface.co/api/datasets/qa994/canifa-captions/commit/main "HTTP/1.1 200 OK"


Da tai ket qua len Hugging Face thanh cong tai: https://huggingface.co/datasets/qa994/canifa-captions
